# PAA KX2

```{device-card} paa-kx2
```

| Property | Value |
|---|---|
| Connection | CANopen, 500 kbit/s; optional serial barcode reader |
| Supported topology | Shoulder, vertical Z, radial elbow, wrist, servo gripper |
| Position units | Millimeters for Z, elbow and jaws; degrees for shoulder and wrist |
| Lifecycle | `setup()` reads calibration; `stop()` restores saved configuration |

## How it talks

The arm's drives share a CAN bus. The barcode reader uses a separate serial connection. Install `pylabrobot[canopen,serial]` and configure the adapter as described in [installation](installation.md). No vendor application should control the arm at the same time.

## Physical setup

Connect the arm's CAN USB connection and power according to its manual. Check the adapter's channel name and 500 kbit/s configuration. Keep the e-stop accessible, clear the arm's travel, and support any load before entering freedrive. Measure the installed gripper's grip-center length and vertical offset; the default zero offsets describe the wrist reference point.

Choose the interface and channel that actually enumerate on your host. This Linux example uses SocketCAN; other adapters use their python-can interface names.

In [ ]:
from pylabrobot.paa import KX2
from pylabrobot.paa.kx2 import Axis

kx2 = KX2(interface="socketcan", channel="can0", bitrate=500000,
          gripper_length=0, gripper_z_offset=0)

## Connect

`setup()` reads the arm's calibration and saves the configuration it changes. It does not automatically home, enable motors or close the jaws. Homing is an explicit operation below. Always finish with `stop()`, including after an exception.

In [ ]:
await kx2.setup()

## Read joint state

The device-local `Axis` keys and `move_to_joint_state` / `request_joint_state` names follow the PreciseFlex convention. The KX2 elbow is a radial slide in millimeters. Its geometry and calibration belong to this driver.

In [ ]:
joint_state = await kx2.request_joint_state()
joint_state

## Read Cartesian pose

The location is the grip center. KX2 yaw is counterclockwise about +Z, with zero pointing along -Y for the barcode-reader finger. Shoulder zero places the radial slide along +Y. Selecting the proximity-sensor finger shifts reported yaw by 180 degrees without moving the grip center. Do not reuse Cartesian poses from another arm without transforming its coordinate frame.

In [ ]:
pose = await kx2.request_gripper_pose()
pose

## Move to an absolute joint state

Use taught, checked targets within this arm's travel limits. Omitted axes are unchanged. This first example commands the measured Z position, making it a no-op until you deliberately change the target. Speed and acceleration settings apply to this call.

In [ ]:
await kx2.move_to_joint_state(
  {Axis.Z: joint_state[Axis.Z]}, max_gripper_speed=20, max_gripper_acceleration=100
)

## Move to a Cartesian pose

The default joint path can curve through space. Check the entire path for clearance. This example commands the measured pose; edit it only after establishing your coordinate frame. Streamed linear motion is described in [continuous motion](continuous-motion.ipynb).

In [ ]:
await kx2.move_to_location(
  pose.location, pose.rotation.z, max_gripper_speed=20, max_gripper_acceleration=100
)

## Home the gripper

Remove any plate and clear the jaws before running this cell. Homing moves to the hard stop and encoder index and updates the jaw calibration. It restores temporary current, homing and motion settings afterwards.

In [ ]:
await kx2.gripper.home()

## Read the gripper

The physical jaws are available as `kx2.gripper`, as on current PreciseFlex drivers. Their travel limits come from this arm's calibration.

In [ ]:
jaw_width = await kx2.gripper.request_jaw_position()
jaw_width

## Position the jaws

Use an absolute width in millimeters. This example leaves the jaws at their measured width. An ordinary coordinated joint move refuses a closing jaw target unless `close_gripper_without_force_sensing=True` is explicitly passed.

In [ ]:
await kx2.gripper.move_to_jaw_position(jaw_width, force_sensing=False)

## Grip a plate

Choose a target within the calibrated jaw travel and a current limit suitable for the plate and tooling. `max_force_percent` is a percentage of motor current, not newtons. `max_speed` and `max_acceleration` limit jaw motion in mm/s and mm/s².

A force-sensed close requires measurable closing travel and plate detection before accepting sustained stationary current as contact. `check_plate_gripped=True` also checks the held grip. The closing target and current limit remain active until release or disconnection. Opening ends the closing target, uses normal opening current, and restores the current limits saved before the grip.

The following cell reads the plate sensor and grip evidence without moving.

In [ ]:
await kx2.gripper.sense_plate_gripped()

## Teach and park

Freedrive disables the selected positioning motors. Support the arm or load before running this cell; choose only axes that can be moved safely by hand.

In [ ]:
await kx2.start_freedrive_mode(free_axes=[Axis.SHOULDER, Axis.ELBOW, Axis.WRIST])

Move the arm by hand to a clear parking pose, then read the positioning joints.

In [ ]:
taught = await kx2.request_joint_state()
kx2.parking_position = {axis: taught[axis] for axis in (Axis.SHOULDER, Axis.Z, Axis.ELBOW, Axis.WRIST)}

Restore the motor enable states saved when freedrive began. This refuses to re-enable after a fault or e-stop.

In [ ]:
await kx2.stop_freedrive_mode()

`park()` uses your configured target and leaves the jaws unchanged. There is no universal default parking pose. Check clearance from the current position before moving.

In [ ]:
await kx2.park()

## Stop an operation

`halt()` cancels arm motion and stops the positioning axes while preserving gripper holding power. It blocks subsequent motion in this connection. Resolve the cause, then stop and reconnect before continuing. Use the physical e-stop for an emergency; host software depends on the CAN connection.

In [ ]:
await kx2.halt()

## Disconnect

`stop()` joins active work, restores saved registers, PDO mappings and node states, and closes the connections. It reports restoration failures. Faulted motors are not automatically re-enabled. If the gripper was disabled when connecting, stop disables it again; remove or support a held plate before disconnecting.

In [ ]:
await kx2.stop()